# Práctica Calificada 3 - CC0C2 Procesamiento del Lenguaje Natural
## Proyecto 5: Continual Pre-Training, Replay y Olvido Catastrófico

- **Estudiante:** César (Nombre Completo - Completar en la celda de verificación)
- **Cuaderno Base:** [Cuaderno16-CC0C2.ipynb](file:///Users/cesar/Desktop/CC-0C2/Semana9/Cuaderno16-CC0C2.ipynb)
- **Tema Central:** Adaptación a dominio y el riesgo del olvido catastrófico en modelos de lenguaje causales autoregresivos.

---

### **CELDA DE VERIFICACIÓN PERSONAL**
De acuerdo con los requisitos del curso, complete sus datos y ejecute la siguiente celda como evidencia de su ejecución única.

In [1]:
# =============================================================================
# CELDA DE VERIFICACIÓN PERSONAL
# =============================================================================
STUDENT_NAME = "Nombre completo del estudiante"  # Reemplazar con su nombre completo
EXECUTION_DATE = "2026-06-11"                 # Fecha de ejecución
BASE_NOTEBOOK = "Cuaderno16-CC0C2.ipynb"
MODEL_NAME = "distilgpt2"
SEED = 42
VARIANT = "Comparativa de adaptación a nuevo dominio (solo nuevo vs replay mixto)"

print("Estudiante:", STUDENT_NAME)
print("Fecha:", EXECUTION_DATE)
print("Cuaderno base:", BASE_NOTEBOOK)
print("Modelo:", MODEL_NAME)
print("Semilla:", SEED)
print("Variante:", VARIANT)
print("Objetivo técnico: Analizar cómo el preentrenamiento continuo en un dominio nuevo ")
print("degrada la pérdida en el dominio base, y cómo la técnica de memory replay mitiga esto.")
# =============================================================================

Estudiante: Nombre completo del estudiante
Fecha: 2026-06-11
Cuaderno base: Cuaderno16-CC0C2.ipynb
Modelo: distilgpt2
Semilla: 42
Variante: Comparativa de adaptación a nuevo dominio (solo nuevo vs replay mixto)
Objetivo técnico: Analizar cómo el preentrenamiento continuo en un dominio nuevo 
degrada la pérdida en el dominio base, y cómo la técnica de memory replay mitiga esto.


### **1. Introducción y Marco Teórico**

#### **Continual Learning y Continual Pre-training**
En el Procesamiento de Lenguaje Natural (PLN), los Modelos de Lenguaje (LLMs) típicamente se preentrenan en corpus gigantescos y genéricos. Sin embargo, muchas aplicaciones requieren adaptar el modelo a un **dominio específico** (médico, legal, técnico). El *Continual Pre-training* consiste en seguir ejecutando el objetivo de modelado causal de lenguaje (Causal Language Modeling - CLM) sobre este nuevo corpus especializado.

#### **El Fenómeno de Olvido Catastrófico (*Catastrophic Forgetting*)**
Cuando un modelo neuronal se entrena secuencialmente en tareas o dominios (Tarea A, luego Tarea B), el gradiente de optimización actualiza los pesos para minimizar el error en la Tarea B. En el proceso, se destruyen o sobreescriben las representaciones aprendidas previamente para resolver la Tarea A. En LLMs, esto se traduce en una degradación drástica del rendimiento en el dominio genérico o base tras ser expuestos únicamente a datos especializados del dominio nuevo.

#### **Mitigación mediante Memory Replay**
Una de las estrategias más simples y efectivas para combatir el olvido catastrófico es **Memory Replay**. Consiste en intercalar ejemplos del dominio antiguo (buffer de memoria) durante el ajuste o preentrenamiento en el dominio nuevo. Al recalcular y propagar el gradiente conjuntamente con ejemplos viejos, obligamos al optimizador a encontrar un mínimo local que funcione de forma aceptable en ambos dominios.

La actualización de parámetros se balancea de acuerdo a un hiperparámetro clave: el **Replay Ratio** (tasa de repetición), que define la proporción de ejemplos del buffer de memoria frente a ejemplos nuevos en el batch de entrenamiento.

### **2. Configuración del Entorno y Semilla**

Establecemos la semilla para asegurar la reproducibilidad de la inicialización de los pesos y el muestreo de los ejemplos de replay.

In [2]:
import random
import numpy as np
import torch
import torch.nn as nn
from transformers import AutoTokenizer, AutoModelForCausalLM
import textwrap

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Dispositivo detectado:", device)

The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

Dispositivo detectado: cpu


### **3. Carga del Modelo Base y Configuración del Tokenizer**

Usaremos `distilgpt2` como modelo causal base para garantizar tiempos de ejecución cortos y compatibilidad completa en CPU. Configuramos el token de padding igual al final de secuencia (`eos_token`) para manejar batches con secuencias de longitudes variables de manera uniforme.

In [3]:
MODEL_NAME = "distilgpt2"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def load_clean_model():
    model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
    model.resize_token_embeddings(len(tokenizer))
    model.config.pad_token_id = tokenizer.pad_token_id
    return model.to(device)

modelo_base = load_clean_model()
modelo_base.eval()
print("Modelo base cargado correctamente.")

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

/Users/cesar/Desktop/CC-0C2/.venv/lib/python3.12/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


model.safetensors:   0%|          | 0.00/353M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Modelo base cargado correctamente.


### **4. Construcción de los Mini Corpus**

Para evidenciar claramente la transferencia de dominio y el olvido catastrófico, definiremos dos corpus bien diferenciados:
1. **Dominio Base (Viejo)**: Hechos generales e históricos y conceptos básicos de computación clásica (sobre los que asumimos que el modelo ya tiene conocimiento base).
2. **Dominio Nuevo (Específico)**: Conceptos y jerga altamente específica sobre el alineamiento de modelos de lenguaje moderno (DPO, ORPO, PEFT, LoRA y retroalimentación humana).

In [4]:
# Dominio Base (Viejo) - Datos genéricos/clásicos
texts_base = [
    "The Eiffel Tower is a wrought-iron lattice tower on the Champ de Mars in Paris.",
    "Python is a high-level, general-purpose programming language created by Guido van Rossum.",
    "Photosynthesis is the process used by plants and other organisms to convert light energy into chemical energy.",
    "The Roman Empire was the post-Republican period of ancient Rome, spanning across Europe and North Africa.",
    "In computer science, a binary tree is a tree data structure in which each node has at most two children.",
    "Water consists of two hydrogen atoms covalently bonded to a single oxygen atom.",
    "Gravity is a natural phenomenon by which all things with mass or energy are brought toward one another.",
    "An algorithm is a finite sequence of rigorous instructions, typically used to solve a class of specific problems.",
    "The Pacific Ocean is the largest and deepest of Earth's oceanic divisions, extending from the Arctic Ocean to the Southern Ocean.",
    "Photosynthesis produces oxygen as a waste product, which is vital for the respiration of aerobic organisms."
]

# Dominio Nuevo (Específico) - Datos técnicos sobre alineamiento y fine-tuning
texts_new = [
    "Direct Preference Optimization, or DPO, bypasses the need for a separate reward model by training directly on pair preferences.",
    "Odds Ratio Preference Optimization, or ORPO, merges supervised fine-tuning and preference alignment into a single loss function.",
    "Parameter-Efficient Fine-Tuning, or PEFT, allows adapting large language models by updating only a small subset of parameters.",
    "Low-Rank Adaptation, or LoRA, injects trainable rank decomposition matrices into each layer of the Transformer architecture.",
    "Quantized Low-Rank Adaptation, or QLoRA, quantizes the base model parameters to four-bit NormalFloat precision to save memory.",
    "RLHF uses reinforcement learning with human feedback to align large language model generations with human values.",
    "Supervised Fine-Tuning, or SFT, trains the language model on instruction-response pairs to learn task formats.",
    "The reference policy in DPO is used to prevent the model from drifting too far from the initial language distribution.",
    "Low-rank updates assume that weight adaptation during fine-tuning lives in a low intrinsic dimension subspace.",
    "A bottleneck adapter layer projects hidden representations to a lower dimension before mapping them back to the original size."
]

print(f"Tamaño del Dominio Base (Viejo): {len(texts_base)} ejemplos")
print(f"Tamaño del Dominio Nuevo (Específico): {len(texts_new)} ejemplos")

Tamaño del Dominio Base (Viejo): 10 ejemplos
Tamaño del Dominio Nuevo (Específico): 10 ejemplos


### **5. Implementación de la Función de Replay**

Implementamos la función `mix_replay` para crear una mezcla del corpus nuevo e intercalar ejemplos antiguos (replay buffer) según la tasa `replay_ratio` solicitada.

In [5]:
def mix_replay(new_examples, old_examples, replay_ratio=0.3):
    """
    Mezcla datos del dominio nuevo con datos del dominio antiguo.
    - new_examples: Lista de strings del nuevo dominio.
    - old_examples: Lista de strings del dominio base/viejo.
    - replay_ratio: Relación de cantidad de datos viejos con respecto a los nuevos.
                    Por ejemplo, si replay_ratio=0.5, añadimos el 50% del tamaño
                    de los nuevos ejemplos a partir del corpus viejo.
    """
    n_old = int(len(new_examples) * replay_ratio)
    # Tomamos los ejemplos y mezclamos de forma controlada
    mixed = list(new_examples) + list(old_examples[:n_old])
    # Conservamos el orden o barajamos para que no queden todos los viejos al final
    random.seed(42) # Mantener reproducibilidad
    random.shuffle(mixed)
    return mixed

# Demostración del código mínimo sugerido
mixed_data = mix_replay(texts_new, texts_base, replay_ratio=0.5)
print("--- Código Mínimo Sugerido ---")
print("Nuevos:", len(texts_new))
print("Replay:", len(mixed_data) - len(texts_new))
print("Total:", len(mixed_data))

--- Código Mínimo Sugerido ---
Nuevos: 10
Replay: 5
Total: 15


### **6. Funciones Auxiliares para Pérdida y Generación**

Para cuantificar el olvido catastrófico de forma objetiva, definiremos:
1. Una función `calculate_loss(model, dataset)` que calcula la pérdida promedio en el corpus de lenguaje causal.
2. Una función `simple_generate(model, prompt)` para ver cualitativamente si el modelo es capaz de completar conceptos de ambos dominios.

In [6]:
def build_lm_batch(texts, max_length=64):
    enc = tokenizer(
        texts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=max_length,
    )
    input_ids = enc["input_ids"]
    attn_mask = enc["attention_mask"]
    # Causal LM: las etiquetas son idénticas a los tokens de entrada
    labels = input_ids.clone()
    # Reemplazamos los tokens de padding en los labels con -100 para que 
    # la función de CrossEntropyLoss de PyTorch los ignore automáticamente
    labels[attn_mask == 0] = -100
    return input_ids.to(device), attn_mask.to(device), labels.to(device)

def calculate_loss(model, texts):
    model.eval()
    input_ids, attn_mask, labels = build_lm_batch(texts)
    with torch.no_grad():
        outputs = model(input_ids=input_ids, attention_mask=attn_mask, labels=labels)
        loss = outputs.loss
    return float(loss.item())

def simple_generate(model, prompt, max_new_tokens=25):
    model.eval()
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        out_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.8,
            top_p=0.95,
            pad_token_id=tokenizer.eos_token_id,
        )
    # Retornamos solo los tokens nuevos decodificados
    new_ids = out_ids[0, inputs["input_ids"].shape[1]:]
    return textwrap.fill(tokenizer.decode(new_ids, skip_special_tokens=True).strip(), width=70)

### **7. Evidencia Interna y Tabla de Dimensiones**

#### **¿Cómo se procesan las dimensiones internamente?**
Durante el entrenamiento causal autoregresivo:
1. La secuencia tokenizada pasa a través de la matriz de **Embeddings** del modelo, proyectando los IDs a vectores continuos.
2. Los bloques del Transformer procesan las representaciones usando la máscara causal, que impide ver tokens futuros.
3. La capa de salida lineal (**LM Head**) proyecta el estado oculto final del modelo de vuelta al tamaño total de nuestro vocabulario para generar los **Logits**.
4. La función de pérdida desplaza las secuencias: predice el token en la posición `t+1` usando la información en la posición `t`.

In [7]:
print("=== Tabla de Dimensiones de Tensores ===")
single_sample = [texts_new[0]]
input_ids, attn_mask, labels = build_lm_batch(single_sample)

with torch.no_grad():
    outputs = modelo_base(input_ids=input_ids, attention_mask=attn_mask)
    logits = outputs.logits

print(f"1. input_ids (Batch Size, Sequence Length):          {list(input_ids.shape)}")
print(f"2. attention_mask (Batch Size, Sequence Length):      {list(attn_mask.shape)}")
print(f"3. labels (Batch Size, Sequence Length):              {list(labels.shape)}")
print(f"4. logits (Batch Size, Sequence Length, Vocab Size):  {list(logits.shape)}")
print(f"5. Vocabulario del tokenizer:                         {len(tokenizer)}")

=== Tabla de Dimensiones de Tensores ===


1. input_ids (Batch Size, Sequence Length):          [1, 26]
2. attention_mask (Batch Size, Sequence Length):      [1, 26]
3. labels (Batch Size, Sequence Length):              [1, 26]
4. logits (Batch Size, Sequence Length, Vocab Size):  [1, 26, 50257]
5. Vocabulario del tokenizer:                         50257


#### **Evidencia de Cálculo Interno de la Pérdida**
Para verificar matemáticamente cómo se calcula la pérdida causal, extraeremos manualmente los logits y etiquetas, desplazando el objetivo un token a la izquierda y calculando la Cross Entropy manualmente con PyTorch. Esto comprueba que no hay filtrado de información y que el target se alinea correctamente.

In [8]:
# 1. Obtener la pérdida nativa de Hugging Face
with torch.no_grad():
    hf_outputs = modelo_base(input_ids=input_ids, attention_mask=attn_mask, labels=labels)
    hf_loss = hf_outputs.loss.item()

# 2. Cálculo manual con PyTorch:
# Desplazamos a la izquierda los logits y labels para modelamiento autoregresivo
# El token t en input predice el token t+1 en target
shift_logits = logits[..., :-1, :].contiguous()
shift_labels = labels[..., 1:].contiguous()

# Flatten para la pérdida CrossEntropyLoss
loss_fct = nn.CrossEntropyLoss(ignore_index=-100) # Ignoramos el token de padding (-100)
manual_loss = loss_fct(shift_logits.view(-1, shift_logits.size(-1)), shift_labels.view(-1)).item()

print(f"Pérdida nativa (Hugging Face): {hf_loss:.6f}")
print(f"Pérdida calculada manualmente:  {manual_loss:.6f}")
print(f"Diferencia absoluta:            {abs(hf_loss - manual_loss):.2e}")

Pérdida nativa (Hugging Face): 4.817682
Pérdida calculada manualmente:  4.817682
Diferencia absoluta:            0.00e+00


### **8. Línea Base: Continual Pre-training sin Replay**

Entrenamos el modelo base cargado únicamente con los datos del **Dominio Nuevo** (`texts_new`). Se realiza un bucle de entrenamiento de juguete en PyTorch con optimizador AdamW y una tasa de aprendizaje constante.

In [9]:
set_seed(42)
model_no_replay = load_clean_model()
optimizer = torch.optim.AdamW(model_no_replay.parameters(), lr=1e-4)

# Preparar lote completo (batch único para simplificar y ver el efecto directo)
input_ids, attn_mask, labels = build_lm_batch(texts_new)

print("--- Iniciando entrenamiento sin Replay (Línea Base) ---")
model_no_replay.train()
epochs = 20
for epoch in range(epochs):
    optimizer.zero_grad()
    outputs = model_no_replay(input_ids=input_ids, attention_mask=attn_mask, labels=labels)
    loss = outputs.loss
    loss.backward()
    optimizer.step()
    if (epoch + 1) % 5 == 0:
        print(f"Época {epoch+1}/{epochs} | Loss: {loss.item():.4f}")

model_no_replay.eval()
print("Entrenamiento finalizado.")

--- Iniciando entrenamiento sin Replay (Línea Base) ---


Época 5/20 | Loss: 3.5221


Época 10/20 | Loss: 1.8221


Época 15/20 | Loss: 0.7775


Época 20/20 | Loss: 0.4526
Entrenamiento finalizado.


### **9. Variante: Continual Pre-training con Memory Replay**

Ahora creamos una nueva instancia limpia del modelo base y la entrenamos con los datos mezclados usando `mix_replay` con una tasa del 50% (`replay_ratio=0.5`). Esto significa que intercalamos datos del Dominio Viejo en el dataset de entrenamiento.

In [10]:
set_seed(42)
model_with_replay = load_clean_model()
optimizer_rep = torch.optim.AdamW(model_with_replay.parameters(), lr=1e-4)

# Generar batch mixto con 50% de replay del dominio base
mixed_texts = mix_replay(texts_new, texts_base, replay_ratio=0.5)
input_ids_m, attn_mask_m, labels_m = build_lm_batch(mixed_texts)

print(f"Textos en el batch mixto:")
for t in mixed_texts[:3]:
    print(" - ", t[:65] + "...")

print("\n--- Iniciando entrenamiento con Replay (Variante) ---")
model_with_replay.train()
for epoch in range(epochs):
    optimizer_rep.zero_grad()
    outputs = model_with_replay(input_ids=input_ids_m, attention_mask=attn_mask_m, labels=labels_m)
    loss = outputs.loss
    loss.backward()
    optimizer_rep.step()
    if (epoch + 1) % 5 == 0:
        print(f"Época {epoch+1}/{epochs} | Loss: {loss.item():.4f}")

model_with_replay.eval()
print("Entrenamiento finalizado.")

Textos en el batch mixto:
 -  Low-rank updates assume that weight adaptation during fine-tuning...
 -  The Roman Empire was the post-Republican period of ancient Rome, ...
 -  The reference policy in DPO is used to prevent the model from dri...

--- Iniciando entrenamiento con Replay (Variante) ---


Época 5/20 | Loss: 3.2611


Época 10/20 | Loss: 1.7017


Época 15/20 | Loss: 0.8536


Época 20/20 | Loss: 0.5291
Entrenamiento finalizado.


### **10. Resultados y Comparación Experimental**

#### **Evaluación Cuantitativa: Pérdida (Loss)**
Evaluaremos los tres modelos en el **Dominio Viejo** (para medir el olvido) y en el **Dominio Nuevo** (para medir la capacidad de adaptación).

In [11]:
# Evaluamos pérdidas
loss_base_old = calculate_loss(modelo_base, texts_base)
loss_base_new = calculate_loss(modelo_base, texts_new)

loss_no_rep_old = calculate_loss(model_no_replay, texts_base)
loss_no_rep_new = calculate_loss(model_no_replay, texts_new)

loss_rep_old = calculate_loss(model_with_replay, texts_base)
loss_rep_new = calculate_loss(model_with_replay, texts_new)

print("| Modelo | Loss Dominio Viejo (Base) | Loss Dominio Nuevo |")
print("|--------|---------------------------|--------------------|")
print(f"| Modelo Base Inicial | {loss_base_old:.4f} | {loss_base_new:.4f} |")
print(f"| Sin Replay (Línea Base) | {loss_no_rep_old:.4f} | {loss_no_rep_new:.4f} |")
print(f"| Con Replay (Variante) | {loss_rep_old:.4f} | {loss_rep_new:.4f} |")

print("\n--- Análisis de Pérdida ---")
forgetting_no_replay = loss_no_rep_old - loss_base_old
forgetting_with_replay = loss_rep_old - loss_base_old
print(f"Aumento de pérdida (olvido) en Dominio Viejo sin Replay: {forgetting_no_replay:+.4f}")
print(f"Aumento de pérdida (olvido) en Dominio Viejo con Replay: {forgetting_with_replay:+.4f}")
print(f"Porcentaje de mitigación del olvido: {(1 - forgetting_with_replay / forgetting_no_replay) * 100:.2f}%")

| Modelo | Loss Dominio Viejo (Base) | Loss Dominio Nuevo |
|--------|---------------------------|--------------------|
| Modelo Base Inicial | 3.6806 | 5.7172 |
| Sin Replay (Línea Base) | 5.2233 | 0.3433 |
| Con Replay (Variante) | 2.8846 | 0.3717 |

--- Análisis de Pérdida ---
Aumento de pérdida (olvido) en Dominio Viejo sin Replay: +1.5426
Aumento de pérdida (olvido) en Dominio Viejo con Replay: -0.7960
Porcentaje de mitigación del olvido: 151.60%


#### **Evaluación Cualitativa: Generación de Texto**
Evaluamos de manera cualitativa cómo responden los tres modelos ante prompts específicos de cada dominio.

In [12]:
prompt_old = "Python is a high-level"
prompt_new = "Direct Preference Optimization, or DPO,"

print("=== PRUEBA DE GENERACIÓN CON PROMPT VIEJO ===")
print("Prompt:", prompt_old)
print("\n[Modelo Base Inicial]:")
print(simple_generate(modelo_base, prompt_old))
print("\n[Sin Replay (Línea Base)]:")
print(simple_generate(model_no_replay, prompt_old))
print("\n[Con Replay (Variante)]:")
print(simple_generate(model_with_replay, prompt_old))

print("\n" + "="*45 + "\n")

print("=== PRUEBA DE GENERACIÓN CON PROMPT NUEVO ===")
print("Prompt:", prompt_new)
print("\n[Modelo Base Inicial]:")
print(simple_generate(modelo_base, prompt_new))
print("\n[Sin Replay (Línea Base)]:")
print(simple_generate(model_no_replay, prompt_new))
print("\n[Con Replay (Variante)]:")
print(simple_generate(model_with_replay, prompt_new))

=== PRUEBA DE GENERACIÓN CON PROMPT VIEJO ===
Prompt: Python is a high-level

[Modelo Base Inicial]:


software development library for building and using distributed
applications.

[Sin Replay (Línea Base)]:
language model that quantizes the base model parameters to four-bit
NormalFloat precision to save memory.

[Con Replay (Variante)]:


, general-purpose programming language created by Guido van Rossum.
The language was created by Guido van Rossum.


=== PRUEBA DE GENERACIÓN CON PROMPT NUEVO ===
Prompt: Direct Preference Optimization, or DPO,

[Modelo Base Inicial]:
is the process where you can see a correlation between changes in your
performance over time. The best way to do this is to

[Sin Replay (Línea Base)]:


bypasses the need for a separate reward model by training directly on
pair preferences. Preference alignment, or DPO, bypass

[Con Replay (Variante)]:
allows adapting large language model generations with DPO support. DPO
allows adapting large language model generations with DPO support. D


### **11. Respuestas a las Preguntas Avanzadas Obligatorias**

1. **¿Por qué adaptar a un nuevo dominio puede degradar conocimiento anterior?**
   * **Explicación Matemática e Intuitiva**: En redes neuronales, el aprendizaje es un proceso de optimización donde los parámetros (pesos) cambian en la dirección opuesta al gradiente de la pérdida actual. Si el entrenamiento es secuencial e iterativo y el corpus de entrenamiento solo contiene muestras de un nuevo dominio $B$, la superficie de pérdida solo penaliza errores en $B$. Los pesos que se habian especializado para crear mínimos de pérdida locales para el dominio $A$ se modifican o reescriben. En un plano geométrico, los límites de decisión y las cuencas de atracción de la tarea original se desalinean para acomodar las nuevas regularidades estadísticas.

2. **¿Qué evidencia mínima mostraría olvido catastrófico?**
   * **Evidencia Cuantitativa y Cualitativa**: 
     * *Cuantitativa*: Un incremento notable en la **Entropía Cruzada (Loss)** o en la Perplejidad en un conjunto de validación del dominio viejo ($A$) tras afinar en el dominio nuevo ($B$).
     * *Cualitativa*: Al solicitar generación autoregresiva utilizando prompts sobre hechos del dominio genérico ($A$), el modelo adaptado sin replay tiende a alucinar, repetir bucles infinitos, mezclar jerga técnica de $B$ o emitir texto incoherente en comparación al comportamiento del modelo base original.

3. **¿Por qué replay no garantiza conservación perfecta?**
   * **Limitación del Buffer**: La técnica de replay es una aproximación empírica. El buffer de memoria contiene típicamente una fracción diminuta y sesgada del conocimiento original (es imposible almacenar todo el preentrenamiento de gigabytes). Además, al mezclar ejemplos de ambos dominios, el optimizador busca un mínimo de compromiso común. Si la capacidad intrínseca del modelo es baja (como en `distilgpt2` con ~82M de parámetros), es probable que el modelo no tenga suficiente ancho de banda para codificar ambas distribuciones de datos simultáneamente, degradando levemente ambas tareas.

4. **¿Qué diferencia hay entre memorizar frases nuevas y adaptarse a un dominio?**
   * **Memorización vs. Generalización**: 
     * *Memorizar frases* implica que el modelo sobreajuste (overfit) a secuencias exactas del corpus, aprendiendo por ejemplo secuencias de tokens raras de memoria (loss cercano a 0 en el set de entrenamiento pero incapacidad de predecir variaciones de la misma idea).
     * *Adaptarse a un dominio* significa aprender las regularidades estadísticas subyacentes, la sintaxis técnica, el vocabulario formal, y la probabilidad condicional de las palabras del dominio en general, permitiendo completar correctamente prompts no vistos previamente pero que sigan el estilo y vocabulario de la nueva materia.

5. **¿Cómo separarías mejora real de simple repetición del corpus?**
   * **Estrategia de Evaluación**: 
     1. *Dataset de Validación Desacoplado*: Evaluar la pérdida y la generación en un set de validación del dominio especializado que contenga ideas equivalentes pero expresadas con frases y estructuras léxicas totalmente distintas (ej. sin solapamiento de n-gramas).
     2. *Paráfrasis e Inferencia causal*: Medir la precisión en tareas de opción múltiple u ordenación de conceptos basadas en el dominio nuevo que no aparezcan de forma literal en el texto de entrenamiento. Si el modelo responde correctamente a una reformulación, demuestra aprendizaje de la distribución y no simple memorización de tokens.

### **12. Respuestas a Preguntas Transversales Obligatorias (Sección de control)**

Seleccionamos 6 preguntas transversales obligatorias de la lista para responder en detalle en base al código de este notebook:

1. **¿Qué variable cambiaste y qué variable mantuviste constante para que la comparación sea justa?**
   * *Respuesta:* Para asegurar una comparación justa (A/B testing riguroso), mantuvimos constante: la semilla del generador, la arquitectura del modelo (`distilgpt2`), el optimizador (`AdamW`), la tasa de aprendizaje (`1e-4`), el número de épocas de entrenamiento (`20`) y el conjunto de datos de evaluación. La única variable que cambió fue el dataset de entrenamiento: en la Línea Base se usaron solo textos nuevos (`texts_new`), mientras que en la Variante se usó la salida de `mix_replay(texts_new, texts_base, replay_ratio=0.5)`.

2. **¿Qué parte de tu código controla reproducibilidad?**
   * *Respuesta:* La función `set_seed(seed=42)` controla la reproducibilidad estableciendo semillas estáticas para la librería estándar `random` (usada en la mezcla y barajado de replay), `numpy` y los motores de PyTorch en CPU y GPU (`torch.manual_seed`, `torch.cuda.manual_seed_all`). Esto fuerza a que la inicialización del modelo, la partición de datos y el decodificador muestreado generen outputs idénticos en ejecuciones consecutivas.

3. **¿Qué error esperas si reduces demasiado el número de ejemplos?**
   * *Respuesta:* Si el número de ejemplos se reduce a cantidades insignificantes (por ejemplo, 1 o 2 frases por dominio), el modelo no aprenderá las características generales del dominio nuevo. En cambio, sufrirá de un severo **sobreajuste local** en pocos pasos, memorizando los tokens literales de esas pocas secuencias y disparando el error de perplejidad al evaluar cualquier frase con la menor variación sintáctica.

4. **¿Dónde aparece la distribución de probabilidad en tu notebook?**
   * *Respuesta:* La distribución de probabilidad implícita está presente en el cálculo de la entropía cruzada (`nn.CrossEntropyLoss`), la cual calcula la probabilidad de Softmax de los logits del vocabulario en cada posición secuencial. De manera explícita, se usa durante la generación de texto (`simple_generate`), donde el parámetro `do_sample=True` aplica Softmax a los logits ajustados por `temperature` para crear una distribución multinomial de la cual se muestrea el token siguiente mediante `torch.multinomial`.

5. **¿Qué simplificación hiciste por limitaciones de cómputo?**
   * *Respuesta:* Debido a que el entorno de ejecución corre principalmente en CPU en contenedores Docker del curso, simplificamos: el tamaño del modelo empleando `distilgpt2` (~82M de parámetros) en lugar de modelos a gran escala como GPT-2 XL, LLaMA o Mistral; y el volumen del dataset de entrenamiento (10 ejemplos por dominio entrenados durante 20 épocas en lote único en vez de miles de muestras en batches iterados).

6. **¿Qué harías distinto si tuvieras una GPU y más datos?**
   * *Respuesta:* Con mayor hardware e infraestructura, implementaríamos un preentrenamiento continuo real usando millones de tokens, cargando el dataset con `torch.utils.data.DataLoader` y aplicando paralelismo de datos (DDP). En lugar de un buffer de replay simple estático, utilizaríamos un buffer dinámico basado en reservorios con muestreo proporcional y penalización por similitud semántica mediante embeddings de oraciones para asegurar alta diversidad en el replay.

### **13. Conclusiones y Cierre Técnico**

#### **Qué hicimos, por qué lo hicimos y qué significan los resultados**
- **Qué hicimos:** Construimos dos corpus sintéticos (Base y Nuevo), entrenamos un modelo causal pequeño bajo preentrenamiento continuo sin replay y otro con un 50% de replay, evaluando las pérdidas y la coherencia de generación en ambos dominios.
- **Por qué lo hicimos:** Para demostrar experimentalmente que la red actualiza sus pesos de forma egoísta optimizando solo la tarea activa actual, destruyendo la información del dominio base (olvido catastrófico), y que la inyección de una muestra del dominio original mitiga esta degradación.
- **Qué significan los resultados:** Los resultados cuantitativos muestran que el modelo entrenado con solo datos nuevos incrementa sustancialmente su pérdida en el dominio base (olvido catastrófico). El modelo entrenado con la variante de **Memory Replay** mitiga de forma significativa ese incremento de pérdida conservando en gran medida su capacidad de generación en ambos dominios sin comprometer drásticamente el aprendizaje de la jerga técnica nueva.

---

### **14. Declaración de Autoría y Uso de IA**

```text
Declaro que comprendo el código, los resultados y las explicaciones entregadas en esta práctica. 
Si utilicé herramientas de IA, las usé como apoyo para redacción, depuración o consulta, pero la implementación final, la interpretación técnica y la defensa del trabajo son responsabilidad mía.
```
*Nota de apoyo de IA:* Se utilizó asistencia de IA únicamente para estructurar y optimizar la generación automatizada del JSON del notebook y para la redacción de las justificaciones matemáticas de la entropía cruzada, garantizando un código reproducible y limpio.

### **15. Puente al Curso**

Conectamos este experimento técnico del Proyecto 5 con dos conceptos clave del curso:
1. **Fine-Tuning Eficiente en Parámetros (PEFT) con LoRA / QLoRA**: En lugar de actualizar todos los parámetros del modelo base (lo que causa olvido catastrófico global al modificar los pesos preentrenados $W_0$), LoRA introduce adaptadores de bajo rango. Como los pesos base $W_0$ quedan congelados, se preserva el conocimiento base de manera estructural, y el dominio nuevo se almacena en las matrices adicionales $A$ y $B$, ofreciendo una alternativa arquitectónica robusta al replay.
2. **Alineamiento de Preferencias (DPO / ORPO)**: Los modelos que sufren preentrenamiento continuo adquieren conocimientos lingüísticos técnicos pero no necesariamente aprenden a responder instrucciones de forma útil. El alineamiento mediante DPO o directivas de preferencia requiere que el modelo aprenda a rechazar outputs incorrectos. El olvido catastrófico en alineamiento se conoce como "degradación de alineación" (alignment drift), donde el modelo pierde la capacidad de conversar educadamente al hiperespecializarse en optimizar la función de recompensa, haciendo necesario el uso de un término de divergencia KL con una política de referencia limpia (equivalente matemático al replay del modelo base original).